# Healthcare Agent Learning Lab 4/10

## Memory, provenance, and stale facts

**Suggested time:** 20 minutes  
**Demonstration type:** Deterministic memory simulation with optional live phrasing

### Learning objectives

- Recognize that agent memory is an engineered external state.
- Identify recency, supersession, status, and provenance requirements.
- Explain how stale memory can produce a confident but unsafe action.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** show that memory is a *design choice*, not a free feature — and that memory which does not track freshness will **resurface a stale fact and drive a confidently wrong action.**

An agent is only as good as the state it remembers. Here the same agent, on the same patient, gives a safe or an unsafe recommendation depending purely on *how* its long-term memory is recalled. (Runs in MOCK mode with no API key.)


> ### In plain terms
> Agents need **memory** to work over time — but memory that doesn't track *how fresh* a fact is can be dangerous. Here it's the **same patient and the same model**; the only thing that changes is *which remembered fact the agent is handed*.
>
> **The point:** one version resurfaces a drug that was stopped weeks ago and gives an unsafe recommendation — the failure is the memory, not the model.
>
> **What you'll see:** two recommendations side by side — "NAIVE recall" vs "CURRENT recall." Read what each one recommends and why they differ.


In [ ]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    """Scripted clinician-assistant reply. The recommendation depends ENTIRELY on
    which memory the agent was given: a stale 'still on warfarin' fact produces a
    confidently wrong (and unsafe) recommendation; the current fact produces the
    right one. A real model behaves the same way — it can only reason over what
    memory hands it."""
    u = " ".join(m["content"] for m in messages if m["role"] == "user")
    if "DISCONTINUED" in u:
        return ("Warfarin was discontinued, so the elevated INR is not warfarin-related: do NOT "
                "adjust or restart warfarin. Flag for clinician review and work up other causes "
                "(liver function, nutrition, interacting meds).")
    return ("INR 4.2 is above the 2.0-3.0 range on warfarin: hold the next warfarin dose and "
            "recheck INR in 2-3 days.")


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### Two kinds of memory
- **Short-term** memory is the context window: what's in front of the model right now.
- **Long-term** memory is an external store the agent writes to and reads back over time.

The store below keeps every fact with the **day** it was recorded, so we can compare two recall strategies later.


In [ ]:
# Long-term memory: an append-only log of facts, each stamped with the day it was written.
MEMORY = []
def remember(kind, value, day):
    MEMORY.append({"kind": kind, "value": value, "day": day})

# Two ways to recall a fact of a given kind:
def recall_naive(kind):
    "First thing we ever wrote — no notion of freshness. (A very common bug.)"
    return next((f for f in MEMORY if f["kind"] == kind), None)

def recall_current(kind):
    "Newest fact of that kind wins — memory that respects time."
    facts = [f for f in MEMORY if f["kind"] == kind]
    return max(facts, key=lambda f: f["day"]) if facts else None


### A timeline where a fact goes stale
The patient starts warfarin, then it is **discontinued** three weeks later. Both events are recorded. Today (day 30) a new INR comes back high.


In [ ]:
remember("anticoagulant", "warfarin 5 mg daily", day=1)                       # started
remember("anticoagulant", "warfarin DISCONTINUED (day 20, bleeding risk)", day=20)  # stopped
TODAY_INR = 4.2   # above the 2.0-3.0 range

for f in MEMORY:
    print(f'  day {f["day"]:>2}: {f["value"]}')


### The agent drafts a recommendation from whatever memory it is handed


In [ ]:
def draft_recommendation(recalled):
    msgs = [{"role":"system","content":"You are a clinical follow-up assistant. Given the anticoagulant "
             "status from memory and the latest INR, recommend the next step in one or two sentences. "
             "Do not change any order yourself."},
            {"role":"user","content":f"Anticoagulant status (from memory): {recalled['value']}. "
             f"Latest INR: {TODAY_INR} (range 2.0-3.0). What is the next step?"}]
    return chat(msgs)

print("NAIVE recall  ->", recall_naive("anticoagulant")["value"])
print("  recommendation:", draft_recommendation(recall_naive("anticoagulant")))
print()
print("CURRENT recall ->", recall_current("anticoagulant")["value"])
print("  recommendation:", draft_recommendation(recall_current("anticoagulant")))


### Takeaway
Both runs used the **same model and the same patient** — only the memory recall differed. Naive recall handed the agent a fact that was true three weeks ago and is now dangerous, and the agent had no way to know. Durable memory is what lets agents work over time, but *stale memory is a first-class failure mode*: recall has to respect recency, supersession, and provenance. In a real system these facts would carry timestamps and status from the record (Synthea / FHIR), and "memory hygiene" — expiring, superseding, and re-verifying — is part of the design, not an afterthought.

*Try:* point the backend at a real open-weight model and re-run — the naive-recall recommendation stays wrong, because the problem is the memory, not the model.


## Exercise

No programming is required for this section.

**Activity:** Review the medication timeline and decide which fact is current, which is superseded, and what should be reverified. Discuss how the conclusion changes when the discontinuation event is missing.

### Discussion prompts

- Which timestamp should govern recall?
- How should the agent behave when two sources conflict?
- When should remembered information expire or be reverified?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: change TODAY and observe which facts should be considered current.
TODAY = 30
MEMORY_TIMELINE = [
    {"day": 1, "fact": "warfarin started", "status": "active"},
    {"day": 21, "fact": "warfarin discontinued", "status": "stopped"},
]
visible = [event for event in MEMORY_TIMELINE if event["day"] <= TODAY]
print("Events known by day", TODAY)
for event in visible:
    print(event)
print("Most recent status:", visible[-1]["status"] if visible else "unknown")


## Optional technical extension

Implement a recall policy using effective time, recorded time, status, and provenance. Add contradictory and late-arriving events and record how conflicts are resolved.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

The failure is caused by recall policy, not necessarily poor language reasoning. Ask participants to distinguish a historical fact from a current fact and an absent fact from a negative fact.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
